# Multiclass data preparation

Adds `subtype` (diagnosis name) and `y_class` (0–6) to the existing Stage1/Stage2 splits so we can train multiclass models (7 lesion types).

**Requires:** `combined_dataset_standardised.csv`, `combined_stage1.csv`, `combined_stage2.csv`, `stage1_train/val/test.csv`, and `Dataset/label_mapping_multiclass.json`.

In [ ]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "Dataset"

standardised_path = DATASET_DIR / "combined_dataset_standardised.csv"
stage1_path = DATASET_DIR / "combined_stage1.csv"
stage2_path = DATASET_DIR / "combined_stage2.csv"
train_path = DATASET_DIR / "stage1_train.csv"
val_path = DATASET_DIR / "stage1_val.csv"
test_path = DATASET_DIR / "stage1_test.csv"
mapping_path = DATASET_DIR / "label_mapping_multiclass.json"

with open(mapping_path) as f:
    label_mapping = json.load(f)
class_to_index = label_mapping["class_to_index"]
print("Class mapping:", class_to_index)

In [ ]:
# Build sample_id in standardised (same as notebook 04)
df_std = pd.read_csv(standardised_path)
df_std["sample_id"] = (
    df_std["dataset_id"].astype(str) + "_" +
    df_std["patient_global"].astype(str) + "_" +
    df_std.index.astype(str)
)
subtype_lookup = df_std[["sample_id", "diagnosis"]].copy()
subtype_lookup = subtype_lookup.rename(columns={"diagnosis": "subtype"})
subtype_lookup["y_class"] = subtype_lookup["subtype"].map(class_to_index)
if subtype_lookup["y_class"].isna().any():
    unmapped = subtype_lookup[subtype_lookup["y_class"].isna()]["subtype"].unique().tolist()
    raise ValueError(f"Unmapped subtypes: {unmapped}")
subtype_lookup["y_class"] = subtype_lookup["y_class"].astype(int)
print("Subtype lookup shape:", subtype_lookup.shape)
print(subtype_lookup["subtype"].value_counts())

In [ ]:
# Stage1 multiclass: merge existing train/val/test with subtype and y_class
for name, path in [("train", train_path), ("val", val_path), ("test", test_path)]:
    df = pd.read_csv(path)
    df = df.merge(subtype_lookup, on="sample_id", how="left")
    if df["y_class"].isna().any():
        raise ValueError(f"Missing y_class in {name}")
    out = DATASET_DIR / f"stage1_multiclass_{name}.csv"
    df.to_csv(out, index=False)
    print(f"Saved {out.name}: {len(df)} rows, {df['y_class'].nunique()} classes")
    print(df["subtype"].value_counts().head())

In [ ]:
# Stage2 multiclass: add subtype and y_class to combined_stage2
stage2 = pd.read_csv(stage2_path)
stage2 = stage2.merge(subtype_lookup, on="sample_id", how="left")
if stage2["y_class"].isna().any():
    raise ValueError("Missing y_class in stage2 after merge")

# Save full combined_stage2_multiclass.csv for Stage2 train (splits by stage1 train/val IDs)
combined_multiclass_path = DATASET_DIR / "combined_stage2_multiclass.csv"
stage2.to_csv(combined_multiclass_path, index=False)
print(f"Saved {combined_multiclass_path.name}: {len(stage2)} rows")

# Also save train/val/test splits for eval and reference
train_ids = set(pd.read_csv(train_path)["sample_id"].astype(str))
val_ids = set(pd.read_csv(val_path)["sample_id"].astype(str))
test_ids = set(pd.read_csv(test_path)["sample_id"].astype(str))
stage2["sample_id"] = stage2["sample_id"].astype(str)

for name, id_set in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    subset = stage2[stage2["sample_id"].isin(id_set)].copy()
    out = DATASET_DIR / f"stage2_multiclass_{name}.csv"
    subset.to_csv(out, index=False)
    print(f"Saved {out.name}: {len(subset)} rows")
    print(subset["subtype"].value_counts().head())